# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


## 🔄 How This Agent Works

Before diving into the code, here's the overall flow I implemented for this assignment:

```
User Query
    ↓
Intent Detection   (look for keywords like "calculate" or "keyword(s)")
    ↓
Tool Selection     (pick the calculator, the keyword extractor, or no tool)
    ↓
Tool Execution     (actually run the selected tool on the query)
    ↓
Structured JSON Response   (always return a dict with "type" and "result")
```

This is a very small, rule-based version of what larger "agentic" systems do: they look at a
request, decide which tool (if any) is needed, call that tool, and package the result nicely.

## 🛠️ Tool 1: Calculator

This tool takes a math expression as a string (e.g. `"20 + 5"`) and evaluates it.
I used Python's built-in `eval()`, but I disabled access to builtins so the expression
can't do anything sneaky like importing modules — it can only do basic arithmetic.

In [1]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result as a string."""
    try:
        # Disabling __builtins__ keeps eval() restricted to just doing math,
        # instead of allowing arbitrary Python code to run.
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception:
        # If the expression is invalid (e.g. "20 + "), fail gracefully.
        return "Error in calculation"

## 🛠️ Tool 2: Keyword Extractor

This tool pulls out the "important" words from a piece of text. My rules:
1. Lowercase everything so "Intelligence" and "intelligence" count as the same word.
2. Only keep alphabetic words (numbers/punctuation are ignored).
3. Only keep words longer than 4 characters (short words like "is", "the" aren't useful keywords).
4. Remove duplicates, but keep the first-seen order.
5. Return at most 5 keywords.

In [2]:
# 🛠️ TOOL 2: Keyword Extractor

import re


def extract_keywords(text: str) -> list:
    """Extract up to 5 unique keywords (alphabetic words longer than 4 letters)."""
    try:
        text_lower = text.lower()

        # \b[a-z]+\b grabs only alphabetic words, ignoring numbers/punctuation.
        words = re.findall(r"[a-z]+", text_lower)

        # Keep words longer than 4 characters.
        long_words = [w for w in words if len(w) > 4]

        # Remove duplicates while preserving the order they first appeared in.
        # (A plain set() would work but loses order, so I build the list manually.)
        seen = set()
        unique_keywords = []
        for word in long_words:
            if word not in seen:
                seen.add(word)
                unique_keywords.append(word)

        return unique_keywords[:5]
    except Exception:
        return []

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

### Intent Detection Logic

Since this is a simple rule-based agent (no ML model), I detect intent by checking
whether certain trigger words appear in the lowercased query:

| Trigger word(s) in query | Detected intent | Tool used |
|---|---|---|
| "calculate" | calculation | `calculator()` |
| "keyword" or "keywords" | keyword extraction | `extract_keywords()` |
| neither | general | no tool, friendly default reply |

In [3]:
# 🤖 AGENT FUNCTION

# This list keeps a record of every query the agent handles, along with
# its response, so we can review the full conversation history later.
query_log = []


def agent(query: str):
    """Route the query to the correct tool and return a structured response."""
    query_lower = query.lower()

    # Step 1: Intent Detection + Step 2: Tool Selection + Step 3: Tool Execution
    if "calculate" in query_lower:
        # Pull out just the math part by removing the trigger word.
        expression = query_lower.replace("calculate", "").strip()
        calculation_result = calculator(expression)

        if calculation_result == "Error in calculation":
            response = {
                "type": "error",
                "result": "Sorry, I couldn't understand that expression."
            }
        else:
            response = {
                "type": "calculation",
                "result": calculation_result
            }

    elif "keyword" in query_lower or "keywords" in query_lower:
        keywords = extract_keywords(query)
        response = {
            "type": "keywords",
            "result": keywords
        }

    else:
        # No matching tool -> general/default response.
        response = {
            "type": "general",
            "result": (
                "I'm a simple AI assistant. I can perform mathematical calculations "
                "and extract keywords from text. Try asking me to calculate an "
                "expression or extract keywords from a sentence."
            )
        }

    # Step 4: Record the query + response in the log before returning it.
    query_log.append({"query": query, "response": response})
    return response


## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

### Expected Output Examples

For the three test queries below, the agent should print something like:

```
Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['extract', 'keywords', 'artificial', 'intelligence', 'transforming']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "I'm a simple AI assistant. I can perform mathematical calculations and extract keywords from text. ..."}
--------------------------------------------------
```

In [4]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['extract', 'keywords', 'artificial', 'intelligence', 'transforming']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "I'm a simple AI assistant. I can perform mathematical calculations and extract keywords from text. Try asking me to calculate an expression or extract keywords from a sentence."}
--------------------------------------------------


## 🎯 Try It Yourself

Run the cell below and type your own queries. Type `exit` when you're done.

In [5]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Query:", user_input)
    print("Response:", agent(user_input))
    print("-" * 50)

Query: calculate 25*8
Response: {'type': 'calculation', 'result': '200'}
--------------------------------------------------
Query: extract keywords from My name is Devansh Mehrotra and I am studying computer science engineering
Response: {'type': 'keywords', 'result': ['extract', 'keywords', 'devansh', 'mehrotra', 'studying']}
--------------------------------------------------


## 📝 Query Log

Every call to `agent()` — from the test cases above and from anything typed into
Interactive Mode — gets recorded in `query_log`. Running the cell below prints out
the full history of queries and their responses, in order.

In [6]:
# 📝 Display the full query/response history

for i, entry in enumerate(query_log, start=1):
    print(f"{i}. Query: {entry['query']}")
    print(f"   Response: {entry['response']}")

1. Query: Calculate 20 + 5
   Response: {'type': 'calculation', 'result': '25'}
2. Query: Extract keywords from Artificial Intelligence is transforming industries
   Response: {'type': 'keywords', 'result': ['extract', 'keywords', 'artificial', 'intelligence', 'transforming']}
3. Query: What is machine learning?
   Response: {'type': 'general', 'result': "I'm a simple AI assistant. I can perform mathematical calculations and extract keywords from text. Try asking me to calculate an expression or extract keywords from a sentence."}
4. Query: calculate 25*8
   Response: {'type': 'calculation', 'result': '200'}
5. Query: extract keywords from My name is Devansh Mehrotra and I am studying computer science engineering
   Response: {'type': 'keywords', 'result': ['extract', 'keywords', 'devansh', 'mehrotra', 'studying']}
